# Sobel Edge Detection

The Sobel operator estimates image gradients using a pair of 3×3 kernels.
Sobel-X responds to **horizontal** intensity changes (detects vertical edges),
Sobel-Y responds to **vertical** changes (detects horizontal edges).
Combining both gives an overall edge-strength map.

## Historical Note

The Sobel operator was introduced by **Irwin Sobel** and **Gary Feldman** in 1968,
presented as a talk at the Stanford Artificial Intelligence Project (SAIL):

> Sobel, I. and Feldman, G. (1968). "A 3×3 Isotropic Gradient Operator for Image Processing."
> Talk at the Stanford Artificial Intelligence Project (SAIL). *Unpublished.*

It was never formally published as a paper. The first published reference appears in:

> Duda, R.O. and Hart, P.E. (1973). *Pattern Classification and Scene Analysis.* Wiley, New York.

Despite its informal origins, the Sobel operator became one of the most widely used
edge detectors in computer vision and remains the standard reference point for
gradient-based edge detection to this day.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.ndimage import correlate as scipy_correlate

## Parameters

Change the settings below to use a different image or noise level.

In [ ]:
IMAGE_PATH = '../data/leaf.jpg'      # path to input image
CROP_SIZE  = 1024                    # centre crop (set to None to skip cropping)
CONTRAST   = 1.4                     # contrast enhancement (1.0 = no change)
BRIGHTNESS = 0.85                    # brightness adjustment  (1.0 = no change)
NOISE_SIGMA = 20                     # std-dev of additive Gaussian noise

BG = '#0E2E32'                       # notebook background colour
DARK_LAYOUT = dict(paper_bgcolor=BG, plot_bgcolor=BG, font_color='white')

# Edge colormaps — dark at zero so edges stand out
CMAP_EDGES_DIV = [                   # diverging: signed gradients (Sobel-X / Sobel-Y)
    [0.0, '#d4702a'],               #   negative → warm orange
    [0.5, '#111111'],               #   zero     → near-black
    [1.0, '#2ec4b6'],               #   positive → teal
]
CMAP_EDGES_SEQ = [                   # sequential: edge magnitude
    [0.0, '#111111'],               #   zero     → near-black
    [0.4, '#1a7a7a'],               #   weak     → dark teal
    [1.0, '#4de8d4'],               #   strong   → bright cyan
]

from matplotlib.colors import LinearSegmentedColormap
MPL_EDGES_DIV = LinearSegmentedColormap.from_list(
    'edges_div', ['#d4702a', '#111111', '#2ec4b6'])

def _square_sync(fig, n_cols):
    """Square pixels on first subplot, synced zoom on the rest."""
    axes = dict(
        yaxis=dict(scaleanchor='x', scaleratio=1, autorange='reversed',
                   constrain='domain'),
        xaxis=dict(constrain='domain'),
    )
    for c in range(2, n_cols + 1):
        axes[f'xaxis{c}'] = dict(constrain='domain', matches='x')
        axes[f'yaxis{c}'] = dict(autorange='reversed', constrain='domain',
                                  matches='y')
    fig.update_layout(**axes)

## 3×3 Sobel Kernels

| Kernel | Detects | Interpretation |
|--------|---------|----------------|
| Sobel-X | Vertical edges | Positive where intensity increases left → right |
| Sobel-Y | Horizontal edges | Positive where intensity increases top → bottom |

In [ ]:
SOBEL_X = np.array([
    [-1, 0, +1],
    [-2, 0, +2],
    [-1, 0, +1]
], dtype=np.float64)

SOBEL_Y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [+1, +2, +1]
], dtype=np.float64)

print('Sobel-X (detects vertical edges):')
print(SOBEL_X)
print(f'Sum of coefficients: {SOBEL_X.sum():.0f}')

print('\nSobel-Y (detects horizontal edges):')
print(SOBEL_Y)
print(f'Sum of coefficients: {SOBEL_Y.sum():.0f}')

In [ ]:
def visualize_kernel(kernel, title, figsize=(4, 4), fontsize=18):
    """Display a convolution kernel as a coloured grid with values."""
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(BG)
    vmax = np.abs(kernel).max()
    ax.imshow(kernel, cmap=MPL_EDGES_DIV, vmin=-vmax, vmax=vmax)
    size = kernel.shape[0]
    for i in range(size + 1):
        ax.axhline(i - 0.5, color='black', linewidth=2)
        ax.axvline(i - 0.5, color='black', linewidth=2)
    for i in range(size):
        for j in range(size):
            val = kernel[i, j]
            text = f'{val:+.0f}' if val != 0 else '0'
            ax.text(j, i, text, ha='center', va='center',
                    fontsize=fontsize, fontweight='bold', color='white')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=14, fontweight='bold', color='white')
    plt.tight_layout()
    plt.show()

visualize_kernel(SOBEL_X, 'Sobel-X Kernel')
visualize_kernel(SOBEL_Y, 'Sobel-Y Kernel')

## Separability of the Sobel Kernel

The Sobel-X kernel is **separable**: it can be written as the outer product
of a column vector and a row vector.

$\displaystyle S_x \;=\; \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \begin{bmatrix} -1 & 0 & +1 \end{bmatrix}$

The column vector $\begin{bmatrix}1 & 2 & 1\end{bmatrix}^T$ is a **smoothing filter**
(weighted average along the vertical direction), and the row vector
$\begin{bmatrix}-1 & 0 & +1\end{bmatrix}$ is a **differencing filter**
(finite difference along the horizontal direction).

So the Sobel-X operator does two things in one pass:
1. **Smooth** vertically (reduce noise in the direction parallel to the edge)
2. **Differentiate** horizontally (detect the intensity change across the edge)

### Order does not matter

The two 1-D convolutions operate along independent axes (rows vs columns),
so they commute:

$\displaystyle (I * \mathbf{c}) * \mathbf{r} \;=\; (I * \mathbf{r}) * \mathbf{c} \;=\; I * S_x$

Smooth-then-differentiate gives the same result as differentiate-then-smooth.

### Why separability matters

A direct 2-D convolution with a $k \times k$ kernel costs $k^2$ multiplications
per pixel. A separable kernel can be applied as two successive 1-D convolutions
(one along rows, one along columns) at a cost of only $2k$ multiplications per pixel.
For a 3×3 kernel this is 6 vs 9 — modest savings.
For larger kernels (e.g. 7×7: 14 vs 49) the speedup is significant.

In [ ]:
# The two 1-D sub-kernels of Sobel-X
smooth_col = np.array([[1], [2], [1]], dtype=np.float64)    # vertical smoothing
diff_row   = np.array([[-1, 0, +1]], dtype=np.float64)      # horizontal differencing

# Verify: outer product reconstructs the full Sobel-X kernel
reconstructed = smooth_col @ diff_row
print("Column vector (smooth):")
print(smooth_col)
print("\nRow vector (differentiate):")
print(diff_row)
print("\nOuter product (column × row):")
print(reconstructed)
print(f"\nMatches Sobel-X? {np.allclose(reconstructed, SOBEL_X)}")

# --- Visualise the sub-kernels side by side with the full kernel ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4),
                         gridspec_kw={'width_ratios': [1, 3, 1, 3]})
fig.patch.set_facecolor(BG)

# Column vector (3×1)
ax = axes[0]
vmax = smooth_col.max()
ax.imshow(smooth_col, cmap=MPL_EDGES_DIV, vmin=-vmax, vmax=vmax, aspect='equal')
for i in range(4):
    ax.axhline(i - 0.5, color='black', linewidth=2)
for j in range(2):
    ax.axvline(j - 0.5, color='black', linewidth=2)
for i in range(3):
    val = smooth_col[i, 0]
    ax.text(0, i, f'{val:+.0f}', ha='center', va='center',
            fontsize=18, fontweight='bold', color='white')
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Smooth\n(column)', fontsize=12, fontweight='bold', color='white')

# Multiplication sign
axes[1].set_facecolor(BG)
axes[1].text(0.5, 0.5, '×', transform=axes[1].transAxes,
             ha='center', va='center', fontsize=40, color='white')
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].spines[:].set_visible(False)

# Row vector (1×3)
ax = axes[2]
vmax_r = np.abs(diff_row).max()
ax.imshow(diff_row, cmap=MPL_EDGES_DIV, vmin=-vmax_r, vmax=vmax_r, aspect='equal')
for i in range(2):
    ax.axhline(i - 0.5, color='black', linewidth=2)
for j in range(4):
    ax.axvline(j - 0.5, color='black', linewidth=2)
for j in range(3):
    val = diff_row[0, j]
    text = f'{val:+.0f}' if val != 0 else '0'
    ax.text(j, 0, text, ha='center', va='center',
            fontsize=18, fontweight='bold', color='white')
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Differentiate\n(row)', fontsize=12, fontweight='bold', color='white')

# Equals sign + full Sobel-X
axes[3].set_facecolor(BG)
axes[3].text(0.08, 0.5, '=', transform=axes[3].transAxes,
             ha='center', va='center', fontsize=40, color='white')
# Inset axes for the full kernel
inset = axes[3].inset_axes([0.2, 0.1, 0.75, 0.8])
vmax_f = np.abs(SOBEL_X).max()
inset.imshow(SOBEL_X, cmap=MPL_EDGES_DIV, vmin=-vmax_f, vmax=vmax_f)
for i in range(4):
    inset.axhline(i - 0.5, color='black', linewidth=2)
    inset.axvline(i - 0.5, color='black', linewidth=2)
for i in range(3):
    for j in range(3):
        val = SOBEL_X[i, j]
        text = f'{val:+.0f}' if val != 0 else '0'
        inset.text(j, i, text, ha='center', va='center',
                   fontsize=16, fontweight='bold', color='white')
inset.set_xticks([])
inset.set_yticks([])
inset.set_title('Sobel-X', fontsize=12, fontweight='bold', color='white')
axes[3].set_xticks([])
axes[3].set_yticks([])
axes[3].spines[:].set_visible(False)

plt.suptitle('Separability: Sobel-X = Smooth (column) × Differentiate (row)',
             fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## The Leaf Image

Same leaf photograph used in the low-pass notebook,
cropped to 1024×1024 and converted to grayscale.

In [ ]:
leaf_rgb = Image.open('../data/leaf.jpg')
leaf_rgb_full = np.array(leaf_rgb)

leaf_gray = leaf_rgb.convert('L')
leaf_gray = ImageEnhance.Contrast(leaf_gray).enhance(1.4)
leaf_gray = ImageEnhance.Brightness(leaf_gray).enhance(0.85)
leaf_array = np.array(leaf_gray)

h, w = leaf_array.shape
crop_size = 1024
sy = (h - crop_size) // 2
sx = (w - crop_size) // 2
leaf_cropped = leaf_array[sy:sy + crop_size, sx:sx + crop_size]
leaf_rgb_cropped = leaf_rgb_full[sy:sy + crop_size, sx:sx + crop_size]

fig = px.imshow(leaf_rgb_cropped, title='Original (colour)')
fig.update_layout(**DARK_LAYOUT, width=600, height=600)
fig.show()

fig = px.imshow(leaf_cropped, color_continuous_scale='gray', zmin=0, zmax=255,
                title='Grayscale')
fig.update_layout(**DARK_LAYOUT, width=600, height=600)
fig.show()

## Cross-Correlation with NumPy

Same sliding-window implementation as in the low-pass notebook.
The outer loop walks over kernel elements; the inner operation is a
full-array multiply-and-accumulate.

### Convolution vs Cross-Correlation

Mathematically, **convolution** flips the kernel 180° before sliding it over the image:

$\displaystyle (f * g)[i,j] \;=\; \sum_{d_i}\sum_{d_j}\; f[\,i - d_i,\; j - d_j\,]\;\cdot\; g[\,d_i,\, d_j\,]$

**Cross-correlation** does *not* flip the kernel:

$\displaystyle (f \star g)[i,j] \;=\; \sum_{d_i}\sum_{d_j}\; f[\,i + d_i,\; j + d_j\,]\;\cdot\; g[\,d_i,\, d_j\,]$

Our implementation uses `padded[di:di+h, dj:dj+w]`, which is the
cross-correlation formula (no flip).
For the **symmetric averaging kernel** (all $\frac{1}{9}$), flipping is a
no-op, so the distinction did not matter.
For **Sobel**, the 180°-flipped kernel is its own negation
(e.g. flipped Sobel-X $= -$Sobel-X), so convolution and cross-correlation
differ only by a sign — the detected edges are identical, only the
gradient-direction convention changes.

In practice, deep-learning frameworks and most image-processing libraries
use cross-correlation and call it "convolution"; the Sobel literature
does the same. We therefore compare against `scipy.ndimage.correlate`.

### The trick: shifted slices

After padding the image, every kernel position `(di, dj)` corresponds to
a particular **neighbour** of each output pixel. The slice
`padded[di:di+h, dj:dj+w]` extracts that neighbour for *all* pixels at once.

For a 3x3 kernel (`kh=kw=3`) with `pad=1`, the padded image has shape
`(h+2, w+2)`. The nine iterations produce:

| `(di, dj)` | Slice | Neighbour |
|---|---|---|
| `(0, 0)` | `padded[0:h, 0:w]` | top-left |
| `(0, 1)` | `padded[0:h, 1:w+1]` | top |
| `(0, 2)` | `padded[0:h, 2:w+2]` | top-right |
| `(1, 0)` | `padded[1:h+1, 0:w]` | left |
| `(1, 1)` | `padded[1:h+1, 1:w+1]` | **centre (pixel itself)** |
| `(1, 2)` | `padded[1:h+1, 2:w+2]` | right |
| `(2, 0)` | `padded[2:h+2, 0:w]` | bottom-left |
| `(2, 1)` | `padded[2:h+2, 1:w+1]` | bottom |
| `(2, 2)` | `padded[2:h+2, 2:w+2]` | bottom-right |

Each slice is an `h x w` array -- every pixel's neighbour in one shot.
Multiply by the corresponding kernel weight and accumulate to get the result.

In [ ]:
def convolve2d(image, kernel, padding='zero'):
    """
    2-D cross-correlation using NumPy array operations.

    Parameters
    ----------
    image : np.ndarray
        Input image (H×W, uint8 or float).
    kernel : np.ndarray
        Convolution kernel (kh×kw).
    padding : str
        'zero'      – pad the input with zeros before correlating.
        'replicate' – pad the input by repeating edge pixels.

    Returns
    -------
    np.ndarray  (float64, same shape as image)
    """
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    h, w = image.shape
    img = image.astype(np.float64)

    if padding == 'replicate':
        padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    else:
        padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant',
                        constant_values=0)

    result = np.zeros((h, w), dtype=np.float64)
    for di in range(kh):
        for dj in range(kw):
            result += kernel[di, dj] * padded[di:di + h, dj:dj + w]

    return result

## Sobel-X: Detecting Vertical Edges

### With Zero Padding

Zero-padding creates artificial edges at the image border —
especially visible with Sobel because the sudden jump
from real pixel values to zero produces a strong gradient response.

In [ ]:
sx_3x3_zero = convolve2d(leaf_cropped, SOBEL_X, padding='zero')

# Compare with scipy.ndimage.correlate (mode='constant' = zero padding)
sx_3x3_zero_ref = scipy_correlate(leaf_cropped.astype(np.float64), SOBEL_X,
                                   mode='constant', cval=0.0)
print(f'Max abs. difference vs scipy.ndimage.correlate: '
      f'{np.abs(sx_3x3_zero - sx_3x3_zero_ref).max():.2e}')

vmax = max(abs(sx_3x3_zero.min()), abs(sx_3x3_zero.max()))
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Grayscale Input',
                                    'Sobel-X — Zero Padding (zoom into borders)'],
                    horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=leaf_cropped, colorscale='gray',
                          zmin=0, zmax=255, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=sx_3x3_zero, colorscale=CMAP_EDGES_DIV,
                          zmin=-vmax, zmax=vmax), row=1, col=2)
_square_sync(fig, 2)
fig.update_layout(**DARK_LAYOUT, width=1200, height=600)
fig.show()

### With Replicate Padding

Replicating edge pixels avoids the artificial border gradient.

In [ ]:
sx_3x3 = convolve2d(leaf_cropped, SOBEL_X, padding='replicate')

sx_3x3_ref = scipy_correlate(leaf_cropped.astype(np.float64), SOBEL_X,
                              mode='nearest')
print(f'Max abs. difference vs scipy.ndimage.correlate: '
      f'{np.abs(sx_3x3 - sx_3x3_ref).max():.2e}')

vmax = max(abs(sx_3x3.min()), abs(sx_3x3.max()))
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Grayscale Input', 'Sobel-X — Replicate Padding'],
                    horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=leaf_cropped, colorscale='gray',
                          zmin=0, zmax=255, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=sx_3x3, colorscale=CMAP_EDGES_DIV,
                          zmin=-vmax, zmax=vmax), row=1, col=2)
_square_sync(fig, 2)
fig.update_layout(**DARK_LAYOUT, width=1200, height=600)
fig.show()

## Sobel-Y: Detecting Horizontal Edges

Applied with replicate padding (the better option).

In [ ]:
sy_3x3 = convolve2d(leaf_cropped, SOBEL_Y, padding='replicate')

sy_3x3_ref = scipy_correlate(leaf_cropped.astype(np.float64), SOBEL_Y,
                              mode='nearest')
print(f'Max abs. difference vs scipy.ndimage.correlate: '
      f'{np.abs(sy_3x3 - sy_3x3_ref).max():.2e}')

vmax = max(abs(sy_3x3.min()), abs(sy_3x3.max()))
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Grayscale Input', 'Sobel-Y — Replicate Padding'],
                    horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=leaf_cropped, colorscale='gray',
                          zmin=0, zmax=255, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=sy_3x3, colorscale=CMAP_EDGES_DIV,
                          zmin=-vmax, zmax=vmax), row=1, col=2)
_square_sync(fig, 2)
fig.update_layout(**DARK_LAYOUT, width=1200, height=600)
fig.show()

## Edge Magnitude

The overall edge strength at each pixel is
$\displaystyle M = |S_x| + |S_y|$
(L1 norm). This combines both gradient directions into a single edge map.

In [ ]:
magnitude_3x3 = np.abs(sx_3x3) + np.abs(sy_3x3)

vmax = magnitude_3x3.max()
fig = make_subplots(rows=1, cols=4,
                    subplot_titles=['Grayscale Input', '|Sobel-X|',
                                    '|Sobel-Y|', 'Edge Magnitude'],
                    horizontal_spacing=0.03)
fig.add_trace(go.Heatmap(z=leaf_cropped, colorscale='gray',
                          zmin=0, zmax=255, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.abs(sx_3x3), colorscale=CMAP_EDGES_SEQ,
                          zmin=0, zmax=vmax, showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=np.abs(sy_3x3), colorscale=CMAP_EDGES_SEQ,
                          zmin=0, zmax=vmax, showscale=False), row=1, col=3)
fig.add_trace(go.Heatmap(z=magnitude_3x3, colorscale=CMAP_EDGES_SEQ,
                          zmin=0, zmax=vmax), row=1, col=4)
_square_sync(fig, 4)
fig.update_layout(**DARK_LAYOUT, height=450, width=1500)
fig.show()

## Effect of Noise on Edge Detection

Sobel amplifies high-frequency content — including noise.
Adding Gaussian noise ($\sigma = 20$) makes the edge map much noisier.

In [ ]:
np.random.seed(42)
noise = np.random.normal(0, 20, leaf_cropped.shape)
leaf_noisy = np.clip(leaf_cropped.astype(np.float64) + noise, 0, 255).astype(np.uint8)

sx_3x3_noisy = convolve2d(leaf_noisy, SOBEL_X, padding='replicate')
sy_3x3_noisy = convolve2d(leaf_noisy, SOBEL_Y, padding='replicate')
magnitude_3x3_noisy = np.abs(sx_3x3_noisy) + np.abs(sy_3x3_noisy)

vmax = max(magnitude_3x3.max(), magnitude_3x3_noisy.max())
fig = make_subplots(rows=1, cols=4,
                    subplot_titles=['Clean Image', 'Clean Edges',
                                    'Noisy Image (σ = 20)', 'Noisy Edges'],
                    horizontal_spacing=0.03)
fig.add_trace(go.Heatmap(z=leaf_cropped, colorscale='gray',
                          zmin=0, zmax=255, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=magnitude_3x3, colorscale=CMAP_EDGES_SEQ,
                          zmin=0, zmax=vmax, showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=leaf_noisy, colorscale='gray',
                          zmin=0, zmax=255, showscale=False), row=1, col=3)
fig.add_trace(go.Heatmap(z=magnitude_3x3_noisy, colorscale=CMAP_EDGES_SEQ,
                          zmin=0, zmax=vmax), row=1, col=4)
_square_sync(fig, 4)
fig.update_layout(**DARK_LAYOUT, height=500, width=1500)
fig.show()